In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Load learned embeddings from Notebook 06
user_embeddings = np.load(
    PROJECT_ROOT / "models" / "user_embeddings.npy"
)

item_embeddings = np.load(
    PROJECT_ROOT / "models" / "item_embeddings.npy"
)

print(f"User embedding shape: {user_embeddings.shape}")
print(f"Item embedding shape: {item_embeddings.shape}")

User embedding shape: (33901, 16)
Item embedding shape: (126, 16)


In [4]:
# Load ratings data
ratings = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "ratings.csv")

# Recreate the same user/item index mappings used for the neural network
user_ids = ratings["user"].unique()
item_ids = ratings["item"].unique()

user_to_index = {
    user_id: index for index, user_id in enumerate(user_ids)
}

item_to_index = {
    item_id: index for index, item_id in enumerate(item_ids)
}

# Map IDs to embedding indices
ratings_nn = ratings.copy()
ratings_nn["user_index"] = ratings_nn["user"].map(user_to_index)
ratings_nn["item_index"] = ratings_nn["item"].map(item_to_index)

print(f"Ratings shape: {ratings_nn.shape}")
print(f"Users mapped: {ratings_nn['user_index'].notna().sum()}")
print(f"Items mapped: {ratings_nn['item_index'].notna().sum()}")

Ratings shape: (233306, 5)
Users mapped: 233306
Items mapped: 233306


In [5]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    ratings_nn,
    test_size=0.20,
    random_state=42
)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

Training samples: 186644
Test samples: 46662


In [6]:
# Get embedding vectors for each interaction
train_user_embeddings = user_embeddings[
    train_data["user_index"].astype(int).to_numpy()
]

train_item_embeddings = item_embeddings[
    train_data["item_index"].astype(int).to_numpy()
]

test_user_embeddings = user_embeddings[
    test_data["user_index"].astype(int).to_numpy()
]

test_item_embeddings = item_embeddings[
    test_data["item_index"].astype(int).to_numpy()
]

# Combine user and item embeddings by element-wise addition
X_train_embeddings = train_user_embeddings + train_item_embeddings
X_test_embeddings = test_user_embeddings + test_item_embeddings

# Rating targets
y_train = train_data["rating"].to_numpy()
y_test = test_data["rating"].to_numpy()

print(f"X_train shape: {X_train_embeddings.shape}")
print(f"X_test shape: {X_test_embeddings.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (186644, 16)
X_test shape: (46662, 16)
y_train shape: (186644,)
y_test shape: (46662,)


In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/Tam-Davies/Projects/ml-course-recommender


In [9]:
from src.models.embedding_models import train_regression_models

regression_models = train_regression_models(
    X_train_embeddings,
    y_train
)

print("Regression models trained:")
for name in regression_models:
    print(f"- {name}")

Regression models trained:
- LinearRegression
- Ridge
- Lasso
- ElasticNet


In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

regression_results = []

for name, model in regression_models.items():
    predictions = model.predict(X_test_embeddings)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)

    regression_results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae
    })

regression_results = (
    pd.DataFrame(regression_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

regression_results

,Model,RMSE,MAE
0,LinearRegression,0.139921,0.062015
1,Ridge,0.139924,0.062006
2,Lasso,0.210945,0.089414
3,ElasticNet,0.210945,0.089414


In [11]:
# Convert ratings into binary class labels
# 0 = rating 2
# 1 = rating 3
y_train_class = (y_train == 3).astype(int)
y_test_class = (y_test == 3).astype(int)

print("Training class distribution:")
print(pd.Series(y_train_class).value_counts().sort_index())

print("\nTest class distribution:")
print(pd.Series(y_test_class).value_counts().sort_index())

Training class distribution:
0      8798
1    177846
Name: count, dtype: int64

Test class distribution:
0     2178
1    44484
Name: count, dtype: int64


In [12]:
from src.models.embedding_models import train_classification_models

classification_models = train_classification_models(
    X_train_embeddings,
    y_train_class
)

print("Classification models trained:")
for name in classification_models:
    print(f"- {name}")

Classification models trained:
- LogisticRegression
- RandomForest


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

classification_results = []

for name, model in classification_models.items():
    predictions = model.predict(X_test_embeddings)

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_test_embeddings)[:, 1]
    else:
        probabilities = predictions

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_class, predictions),
        "Precision": precision_score(y_test_class, predictions, zero_division=0),
        "Recall": recall_score(y_test_class, predictions, zero_division=0),
        "F1": f1_score(y_test_class, predictions, zero_division=0),
        "AUC": roc_auc_score(y_test_class, probabilities),
    })

classification_results = pd.DataFrame(classification_results)

classification_results

,Model,Accuracy,Precision,Recall,F1,AUC
0,LogisticRegression,0.993635,0.994871,0.998471,0.996668,0.986744
1,RandomForest,0.994685,0.995364,0.999078,0.997218,0.991852


In [14]:
# Save embedding regression results
regression_results.to_csv(
    PROJECT_ROOT / "data" / "processed" / "embedding_regression_results.csv",
    index=False
)

# Save embedding classification results
classification_results.to_csv(
    PROJECT_ROOT / "data" / "processed" / "embedding_classification_results.csv",
    index=False
)

print("Embedding model results saved successfully.")

Embedding model results saved successfully.
